# task 3 -> group relative policy optimization

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.data import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides CONFIG for the smoke test, the defaults are the real run
CONFIG = 'configs/grpo.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir'])
FIGURES = RESULTS / 'figures'
STANDARD = cfg['output']
MIDPOINT = cfg['paths']['grpo_midpoint_policy']
FORKS = {'grpo': 'fork_grpo', 'dr_grpo': 'fork_dr_grpo'} #loss type -> run name, as in compare_normalization.py

#fixed colours -> one per difficulty bin in the group-size study, one per loss type in the normalization study
BIN_COLORS = {'all': '#52514e', 'hard': '#eb6834', 'easy': '#2a78d6'}
LOSS_COLORS = {'grpo': '#2a78d6', 'dr_grpo': '#eb6834'}
MUTED = '#52514e'

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'lines.linewidth': 2})


def run(command):
    #runs one of the release's entry points as its own process, exactly as it would be typed in a terminal.
    #each script loads its own models and writes its results to disk, nothing is passed through notebook state
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'


def load_result(name):
    return json.loads((RESULTS / name).read_text())


def frame(name):
    return pd.DataFrame(read_jsonl(RESULTS / name))

## standard grpo continuation ->

20 updates from the supplied midpoint, K = 4 completions per prompt, max-length completions masked from the loss

In [ ]:
run(f'{PYTHON} -m task3_grpo.continue_train --config {CONFIG} --run-name standard --output {STANDARD}')

held-out evaluation of the frozen reference, the supplied midpoint and the continued policy, same prompts and decoding

In [ ]:
run(f'{PYTHON} -m task3_grpo.evaluate --config {CONFIG} --name reference')
run(f'{PYTHON} -m task3_grpo.evaluate --config {CONFIG} --adapter {MIDPOINT} --name midpoint')
run(f'{PYTHON} -m task3_grpo.evaluate --config {CONFIG} --adapter {STANDARD} --name standard')

## group-size study ->

the cached completions regrouped into K = 2 / 4 / 8 at the same total of 192 generations, no model involved

In [ ]:
run(f'{PYTHON} -m task3_grpo.analyze_group_size --config {CONFIG}')

## length-normalization study ->

two short forks from the same midpoint, canonical grpo (divide by T_k) against dr. grpo (divide by a constant)

In [ ]:
run(f'{PYTHON} -m task3_grpo.compare_normalization --config {CONFIG}')

## standard continuation diagnostics

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
standard_log = frame('train_log_standard.jsonl')
standard_summary = load_result('train_summary_standard.json')

#the compute the manual asks to be reported for the standard continuation
display(pd.DataFrame([{
    'gpu': standard_summary['gpu'],
    'updates': standard_summary['updates'],
    'completions_per_prompt': standard_summary['num_generations'],
    'generated_tokens': standard_summary['generated_tokens'],
    'wall_clock_min': standard_summary['wall_clock_s'] / 60,
    'peak_vram_gb': standard_summary['peak_vram_gb'],
}]).round(2))

columns = ['update', 'reward', 'kl', 'group_reward_std', 'uninformative_fraction', 'policy_loss', 'grad_norm', 'entropy',
           'response_length', 'truncated', 'clip_fraction', 'step_kl']
standard_log[columns].to_csv(RESULTS / 'standard_trajectory.csv', index=False)
display(standard_log[columns].round(4))
print('mean within-group reward std', round(standard_log['group_reward_std'].mean(), 4),
      '  uninformative groups', f"{standard_log['uninformative_fraction'].mean():.2%}",
      '  completions masked for hitting the cap', f"{standard_log['truncated'].mean():.2%}")

In [ ]:
#one panel per logged quantity, each on its own scale. every update is one prompt with K completions, so the thin
#line is the raw per-update value and the thick line a 5 update rolling mean to show the trend
panels = [('reward', 'learned reward (mean of K)'), ('kl', 'kl from reference (per token)'),
          ('group_reward_std', 'within-group reward std'), ('policy_loss', 'policy loss'),
          ('grad_norm', 'gradient norm'), ('entropy', 'policy entropy'),
          ('response_length', 'response length (tokens)'), ('truncated', 'fraction of the group masked')]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for axis, (column, title) in zip(axes.ravel(), panels):
    axis.plot(standard_log['update'], standard_log[column], color='#9ec5f4', linewidth=1.2)
    axis.plot(standard_log['update'], standard_log[column].rolling(5, min_periods=1).mean(), color='#2a78d6')
    axis.set_title(title)
    axis.set_xlabel('update')
    axis.set_xticks(range(0, len(standard_log) + 1, 5))
fig.tight_layout()
fig.savefig(FIGURES / 'standard_trajectories.png', dpi=200)
plt.show()

In [ ]:
#held-out behaviour before and after the continuation, with the frozen reference as the anchor
def heldout_row(name):
    ev = load_result(f'eval_{name}.json')
    return {'policy': name, 'heldout_reward': ev['reward_mean'], 'heldout_kl_token': ev['kl_token_mean'],
            'heldout_kl_sequence': ev['kl_sequence_mean'], 'heldout_entropy': ev['entropy_mean'],
            'heldout_length': ev['length_mean'], 'heldout_length_std': ev['length_std'], 'heldout_truncation': ev['truncation_rate']}

heldout = pd.DataFrame([heldout_row(name) for name in ['reference', 'midpoint', 'standard']])
heldout.to_csv(RESULTS / 'standard_heldout.csv', index=False)
display(heldout.round(4))

## group-size study results

In [ ]:
#difficulty rule (defined once, in analyze_group_size.py) -> a prompt's mean reward over its 8 cached completions,
#below the median of the 24 prompts = hard, otherwise easy
group_study = pd.read_csv(RESULTS / 'group_size_study.csv')
display(group_study.round(4))

panels = [('informative_rate', 'informative-group rate'), ('mean_group_reward_std', 'mean within-group reward std'),
          ('relative_signal_variance', 'variance of the group-relative signal'), ('sign_agreement', 'advantage sign agreement with the K = 8 baseline')]
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.2))
for axis, (column, title) in zip(axes, panels):
    for label in ['all', 'hard', 'easy']:
        rows = group_study[group_study['difficulty'] == label].sort_values('K')
        axis.plot(rows['K'], rows[column], color=BIN_COLORS[label], marker='o', markersize=7, label=label)
    axis.set_xscale('log', base=2)
    axis.set_xticks([2, 4, 8], ['2', '4', '8'])
    axis.minorticks_off()
    axis.set_xlabel('group size K (192 generations in every case)')
    axis.set_title(title)
axes[0].legend(frameon=False, title='prompt difficulty')
fig.tight_layout()
fig.savefig(FIGURES / 'group_size_study.png', dpi=200)
plt.show()

## normalization comparison results

In [ ]:
normalization = pd.read_csv(RESULTS / 'normalization_comparison.csv')
display(normalization.set_index('loss_type').T)

#the length-conditioned statistic -> each completion's own gradient norm per unit of |advantage|, against its length.
#only completions that carried gradient are shown (not masked, informative group). canonical grpo gives every completion
#the same total weight, dr. grpo gives every token the same weight, so its completions should climb with length
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
for axis, (loss_type, name) in zip(axes, FORKS.items()):
    completions = frame(f'train_completions_{name}.jsonl')
    active = completions[completions['gradient_norm'] > 0]
    axis.scatter(active['completion_tokens'], active['gradient_norm'] / active['advantage'].abs(), s=70,
                 color=LOSS_COLORS[loss_type], edgecolor='#fcfcfb', linewidth=1.5)
    axis.set_title(f'{loss_type}  ({len(active)} of {len(completions)} completions carried gradient)')
    axis.set_xlabel('completion length (tokens)')
    axis.set_ylabel('gradient norm / |advantage|')
fig.tight_layout()
fig.savefig(FIGURES / 'normalization_gradient_vs_length.png', dpi=200)
plt.show()

In [ ]:
#training trajectories of the two forks, same prompt at the same update
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for axis, (column, title) in zip(axes, [('reward', 'learned reward'), ('kl', 'kl from reference (per token)'),
                                        ('response_length', 'response length (tokens)'), ('grad_norm', 'gradient norm')]):
    for loss_type, name in FORKS.items():
        log = frame(f'train_log_{name}.jsonl')
        axis.plot(log['update'], log[column], color=LOSS_COLORS[loss_type], marker='o', markersize=5, label=loss_type)
    axis.set_title(title)
    axis.set_xlabel('update')
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / 'normalization_fork_trajectories.png', dpi=200)
plt.show()

#held-out length distribution of the two forks next to the midpoint they started from
lengths = pd.DataFrame({name: frame(f'eval_{name}_generations.jsonl')['response_tokens'].describe()
                        for name in ['midpoint', *FORKS.values()]})
display(lengths.round(1))

## qualitative candidates ->

shortlists to read through and pick the report examples from, nothing here is selected automatically

In [ ]:
pd.set_option('display.max_colwidth', 400)

#group informativeness -> the cached prompts with the smallest and the largest reward spread across their 8
#completions. the first are prompts where the group says almost nothing, the second where it separates completions well
prompts = pd.read_csv(RESULTS / 'group_size_prompts.csv').sort_values('reward_std_k8')
print('least informative cached prompts')
display(prompts.head(4))
print('most informative cached prompts')
display(prompts.tail(4))

#the actual completions of the single least and most informative prompt, best and worst reward of each
cache = pd.DataFrame(read_jsonl(cfg['group_cache']))
for label, prompt_id in [('least informative', prompts.iloc[0]['prompt_id']), ('most informative', prompts.iloc[-1]['prompt_id'])]:
    group = cache[cache['prompt_id'] == prompt_id].sort_values('reward')
    print(label)
    display(group.iloc[[0, -1]][['generation_index', 'reward', 'completion_tokens', 'clipped_at_max', 'completion']])

In [ ]:
#normalization -> in each fork, the completions that carried the largest and the smallest gradient per unit advantage,
#with their lengths. shows which completions each normalization lets dominate an update
for loss_type, name in FORKS.items():
    completions = frame(f'train_completions_{name}.jsonl')
    active = completions[completions['gradient_norm'] > 0].copy()
    active['gradient_per_advantage'] = active['gradient_norm'] / active['advantage'].abs()
    print(loss_type)
    display(active.sort_values('gradient_per_advantage', ascending=False)
            [['update', 'completion_tokens', 'reward', 'advantage', 'gradient_per_advantage', 'completion']].iloc[[0, 1, -2, -1]])